# Simulation de Trempe d'un Cylindre en Acier AISI 1045
## Methode des Volumes Finis (MVF) -- Schema Explicite 1D Radial

| | |
|---|---|
| **Auteur** | Khaled Merini -- M1 Mecanique Energetique |
| **Module** | MVF -- Encadrant : Prof. Chikh |
| **Etablissement** | USTHB -- Faculte de Genie Mecanique et des Procedes |
| **Annee** | 2025 / 2026 |

---

### Contexte
La trempe consiste a chauffer un cylindre d'acier AISI 1045 jusqu'a **850 degC**, puis a le plonger dans un fluide a **40 degC**.

### Deux contraintes du cahier des charges
| Contrainte | Critere |
|---|---|
| Metallurgique | Vitesse de refroidissement au centre >= **30 degC/s** entre 800 degC et 500 degC |
| Mecanique | Ecart thermique maximal DeltaT_max <= **200 degC** |


## 1. Importation des bibliotheques

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.4,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

print('Bibliotheques chargees avec succes.')


## 2. Parametres physiques et proprietes de l'AISI 1045

Les proprietes thermiques sont **dependantes de la temperature** et interpolees lineairement a partir de tables tabulees.

In [ ]:
# --- Geometrie ---
R   = 0.025    # Rayon du cylindre [m]

# --- Proprietes du materiau ---
rho = 7800.0   # Masse volumique [kg/m3]

# --- Conditions aux limites ---
T_INIT = 850.0  # Temperature initiale uniforme [degC]
T_INF  = 40.0   # Temperature du fluide de trempe [degC]

# --- Proprietes tabulees de l'AISI 1045 ---
T_TABLE  = np.array([20,  200, 400, 600, 700,  750,  800, 850], dtype=float)
K_TABLE  = np.array([51,   48,  42,  35,  30,   28,   27,  27], dtype=float)  # [W/m.K]
CP_TABLE = np.array([470, 520, 610, 750, 900, 1400,  680, 650], dtype=float)  # [J/kg.K]

print(f'Rayon      : {R*1000:.0f} mm')
print(f'T initiale : {T_INIT} degC')
print(f'T fluide   : {T_INF} degC')
print(f'Masse vol. : {rho} kg/m3')


## 3. Fonctions d'interpolation des proprietes thermiques

Conductivite et capacite calorifique interpolees **lineairement** entre les points tabules a chaque noeud et a chaque pas de temps.

In [ ]:
def k_interp(T):
    """Conductivite thermique k(T) [W/m.K] -- interpolation lineaire."""
    return np.interp(T, T_TABLE, K_TABLE)

def cp_interp(T):
    """Capacite calorifique cp(T) [J/kg.K] -- interpolation lineaire."""
    return np.interp(T, T_TABLE, CP_TABLE)

# --- Verification ---
print(f'  T [degC] | k [W/m.K] | cp [J/kg.K]')
print('  ' + '-'*38)
for T in [100, 500, 800]:
    print(f'  {T:>7} | {k_interp(T):>9.1f} | {cp_interp(T):>11.0f}')

# --- Visualisation des proprietes ---
T_plot = np.linspace(20, 850, 200)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(T_plot, k_interp(T_plot), color='#2980b9', linewidth=2)
ax1.scatter(T_TABLE, K_TABLE, color='#e74c3c', zorder=5, label='Points tabules')
ax1.set_xlabel('Temperature [degC]')
ax1.set_ylabel('k [W/m.K]')
ax1.set_title('Conductivite thermique k(T)')
ax1.legend()

ax2.plot(T_plot, cp_interp(T_plot), color='#27ae60', linewidth=2)
ax2.scatter(T_TABLE, CP_TABLE, color='#e74c3c', zorder=5, label='Points tabules')
ax2.set_xlabel('Temperature [degC]')
ax2.set_ylabel('cp [J/kg.K]')
ax2.set_title('Capacite calorifique cp(T)')
ax2.legend()

plt.suptitle('Proprietes thermiques de l\'AISI 1045', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('properties_AISI1045.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Solveur MVF -- Discretisation et integration temporelle

### Equation gouvernante (cylindrique 1D radiale)

$$\\rho c_p(T)\\,\\frac{\\partial T}{\\partial t} = \\frac{1}{r}\\frac{\\partial}{\\partial r}\\!\\left(r\\,k(T)\\,\\frac{\\partial T}{\\partial r}\\right)$$

### Conditions aux limites
- **Centre** $(r=0)$ : symetrie axiale -> flux nul
- **Surface** $(r=R)$ : convection -> $-k\\,\\partial T/\\partial r = h(T_{surface} - T_\\infty)$

Les conductivites aux interfaces sont calculees par **moyenne harmonique** :
$k_e = \\dfrac{2 k_P k_E}{k_P + k_E}$

In [ ]:
def run_simulation(N, h, t_max=60.0):
    """
    Simule la trempe par la Methode des Volumes Finis (schema explicite).

    Parametres
    ----------
    N     : int   -- Nombre de volumes de controle
    h     : float -- Coefficient de convection [W/m2.K]
    t_max : float -- Duree de simulation [s]

    Retourne
    --------
    dict avec : r, dt, time, T_center, T_surface, delta_T,
                delta_T_max, cooling_rate, profiles
    """
    # -- Maillage : N volumes de controle annulaires uniformes --
    dr = R / N
    r  = np.linspace(dr / 2, R - dr / 2, N)  # Centres des volumes [m]

    # -- Pas de temps stable (critere de Fourier, Fo <= 0.4) --
    alpha_max = np.max(K_TABLE / (rho * CP_TABLE))
    dt = 0.4 * dr**2 / alpha_max

    # -- Initialisation --
    T    = np.full(N, T_INIT)
    time = 0.0

    # -- Stockage --
    time_hist, T_cen_hist, T_sur_hist, dT_hist = [], [], [], []
    saved_profiles = {}
    saved_times    = [1, 5, 10, 30, 60]

    # == Boucle temporelle ====================================
    while time < t_max:
        T_new = T.copy()

        # -- Noeud central (i=0) : symetrie, flux_w = 0 --
        re  = dr / 2
        ke  = 2.0 * k_interp(T[0]) * k_interp(T[1]) / (k_interp(T[0]) + k_interp(T[1]))
        T_new[0] = T[0] + dt * ke * (2*np.pi*re) * (T[1]-T[0]) / (
                   dr * rho * cp_interp(T[0]) * np.pi*re**2)

        # -- Noeuds internes (i=1 .. N-2) --
        for i in range(1, N - 1):
            rw, re = r[i]-dr/2, r[i]+dr/2
            Aw, Ae = 2*np.pi*rw, 2*np.pi*re
            V  = np.pi * (re**2 - rw**2)
            kw = 2.0*k_interp(T[i])*k_interp(T[i-1]) / (k_interp(T[i])+k_interp(T[i-1]))
            ke = 2.0*k_interp(T[i])*k_interp(T[i+1]) / (k_interp(T[i])+k_interp(T[i+1]))
            fw = kw * Aw * (T[i] - T[i-1]) / dr
            fe = ke * Ae * (T[i+1] - T[i]) / dr
            T_new[i] = T[i] + dt * (fe - fw) / (rho * cp_interp(T[i]) * V)

        # -- Noeud de surface (i=N-1) : convection --
        i  = N - 1
        rw, re = r[i]-dr/2, R
        Aw, Ae = 2*np.pi*rw, 2*np.pi*re
        V  = np.pi * (re**2 - rw**2)
        kw = 2.0*k_interp(T[i])*k_interp(T[i-1]) / (k_interp(T[i])+k_interp(T[i-1]))
        fw = kw * Aw * (T[i] - T[i-1]) / dr
        fc = h  * Ae * (T_INF - T[i])          # Flux convection
        T_new[i] = T[i] + dt * (fc - fw) / (rho * cp_interp(T[i]) * V)

        T    = T_new.copy()
        time += dt

        # -- Enregistrement --
        time_hist.append(time)
        T_cen_hist.append(T[0])
        T_sur_hist.append(T[-1])
        dT_hist.append(T[0] - T[-1])

        for ts in saved_times:
            if ts not in saved_profiles and time >= ts:
                saved_profiles[ts] = T.copy()

    # == Post-traitement ======================================
    time_arr = np.array(time_hist)
    T_c_arr  = np.array(T_cen_hist)
    T_s_arr  = np.array(T_sur_hist)
    dT_arr   = np.array(dT_hist)

    # Vitesse de refroidissement au centre entre 800 degC et 500 degC
    i800 = np.argmin(np.abs(T_c_arr - 800))
    i500 = np.argmin(np.abs(T_c_arr - 500))
    cooling_rate = (800 - 500) / (time_arr[i500] - time_arr[i800])  # [degC/s]

    return {
        'r': r, 'dt': dt, 'time': time_arr,
        'T_center': T_c_arr, 'T_surface': T_s_arr,
        'delta_T': dT_arr, 'delta_T_max': dT_arr.max(),
        'cooling_rate': cooling_rate, 'profiles': saved_profiles
    }

print('Solveur MVF pret.')


## 5. Partie A -- Etude d'independance du maillage

On teste trois discretisations (N = 10, 20, 40 noeuds) pour verifier que les resultats ne dependent plus du maillage a partir d'un certain N.

In [ ]:
h_ref  = 1000
N_list = [10, 20, 40]
mesh_results = {}

print(f"{'N':>5} | {'dt [ms]':>9} | {'Vitesse [degC/s]':>17} | {'DT_max [degC]':>14}")
print('-' * 55)

for N in N_list:
    res = run_simulation(N=N, h=h_ref)
    mesh_results[N] = res
    print(f"{N:>5} | {res['dt']*1000:>9.3f} | {res['cooling_rate']:>17.2f} | {res['delta_T_max']:>14.1f}")

ecart = abs(mesh_results[40]['T_center'][-1] - mesh_results[20]['T_center'][-1]) / mesh_results[40]['T_center'][-1] * 100
print(f'\nEcart relatif T_centre (N=20 vs N=40) = {ecart:.3f} %')
print('-> Maillage independant a N = 40 (ecart < 1 %)')

fig, ax = plt.subplots(figsize=(8, 5))
for N, col in zip(N_list, ['#e74c3c', '#f39c12', '#27ae60']):
    ax.plot(mesh_results[N]['time'], mesh_results[N]['T_center'],
            label=f'N = {N}', color=col, linewidth=2)
ax.set_xlabel('Temps [s]')
ax.set_ylabel('Temperature au centre [degC]')
ax.set_title("Etude d'independance du maillage")
ax.legend()
plt.tight_layout()
plt.savefig('A_grid_independence.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Simulation principale (N = 40, h = 1000 W/m2.K)

Le maillage valide (N = 40) est utilise pour toutes les analyses suivantes.

In [ ]:
res = mesh_results[40]  # Reutilisation du resultat deja calcule

print(f"Vitesse de refroidissement [800->500 degC] : {res['cooling_rate']:.2f} degC/s")
print(f"DeltaT_max centre-surface                  : {res['delta_T_max']:.1f} degC")
print(f"Pas de temps dt                            : {res['dt']*1000:.3f} ms")


## 7. Partie B -- Historiques thermiques au centre et en surface

La surface (r = R) refroidit immediatement car elle est en contact direct avec le fluide. Le centre (r = 0) reste a 850 degC pendant quelques secondes a cause de l'**inertie thermique** de la piece.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(res['time'], res['T_center'],  label='Centre  (r = 0)', color='#2980b9', linewidth=2)
ax.plot(res['time'], res['T_surface'], label='Surface (r = R)', color='#e67e22', linewidth=2)
ax.axhline(800, linestyle='--', color='gray', linewidth=1, alpha=0.7, label='800 degC')
ax.axhline(500, linestyle='--', color='gray', linewidth=1, alpha=0.7, label='500 degC')
ax.set_xlabel('Temps [s]')
ax.set_ylabel('Temperature [degC]')
ax.set_title('Partie B -- Evolution de la temperature au centre et en surface')
ax.legend()
plt.tight_layout()
plt.savefig('B_temperature_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Vitesse de refroidissement (800->500 degC) : {res['cooling_rate']:.2f} degC/s")


## 8. Partie C -- Indicateur de contrainte thermique DeltaT(t)

$$\\Delta T(t) = T_{centre}(t) - T_{surface}(t)$$

Le **pic de DeltaT** represente le moment le plus critique de la trempe. Si DeltaT_max > 200 degC, la piece risque de se fissurer.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(res['time'], res['delta_T'], color='#8e44ad', linewidth=2,
        label='DeltaT = T_centre - T_surface')
ax.axhline(200, linestyle='--', color='red', linewidth=1.5,
           label='Limite DeltaT_max = 200 degC')
ax.fill_between(res['time'], res['delta_T'],
                where=(res['delta_T'] > 200),
                color='red', alpha=0.15, label='Zone hors limite')
ax.set_xlabel('Temps [s]')
ax.set_ylabel('DeltaT [degC]')
ax.set_title('Partie C -- Gradient thermique centre-surface')
ax.legend()
plt.tight_layout()
plt.savefig('C_thermal_gradient.png', dpi=150, bbox_inches='tight')
plt.show()

statut = 'RESPECTEE' if res['delta_T_max'] <= 200 else 'DEPASSEE'
print(f"DeltaT_max = {res['delta_T_max']:.1f} degC  ->  Contrainte mecanique : {statut}")


## 9. Profils de temperature le long du rayon

Ces profils montrent comment la chaleur se propage du centre vers la surface. A t = 1 s, seule la surface a refroidi. Avec le temps, les profils prennent une forme parabolique caracteristique.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors_p = plt.cm.plasma(np.linspace(0.1, 0.85, 5))

for ts, col in zip([1, 5, 10, 30, 60], colors_p):
    if ts in res['profiles']:
        ax.plot(res['r'] * 1000, res['profiles'][ts],
                label=f't = {ts} s', color=col, linewidth=2)

ax.set_xlabel('Rayon [mm]')
ax.set_ylabel('Temperature [degC]')
ax.set_title('Profils de temperature le long du rayon')
ax.legend()
plt.tight_layout()
plt.savefig('radial_profiles.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Partie D -- Analyse parametrique : effet du coefficient h

On fait varier h pour simuler differents fluides et regimes d'agitation :

| Fluide / Regime | h [W/m2.K] |
|---|---|
| Huile | ~500 |
| Eau stagnante | ~1000 |
| Eau agitee moderement | ~3000 |
| Eau agitee violemment | ~5000 |


In [ ]:
h_values   = [500, 1000, 3000, 5000]
cool_rates = []
dT_maxs    = []

print(f"{'h [W/m2.K]':>12} | {'Vitesse [dC/s]':>14} | {'DT_max [dC]':>12} | {'Metallurgie':>12} | {'Mecanique':>10}")
print('-' * 70)

for h in h_values:
    r_h = run_simulation(N=40, h=h)
    cool_rates.append(r_h['cooling_rate'])
    dT_maxs.append(r_h['delta_T_max'])
    m = 'OK >=30' if r_h['cooling_rate'] >= 30 else 'NOK <30'
    s = 'OK <=200' if r_h['delta_T_max'] <= 200 else 'NOK >200'
    print(f"{h:>12} | {r_h['cooling_rate']:>14.1f} | {r_h['delta_T_max']:>12.1f} | {m:>12} | {s:>10}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(h_values, cool_rates, 'o-', color='#2980b9', linewidth=2, markersize=8)
ax1.axhline(30, linestyle='--', color='red', linewidth=1.5, label='Min = 30 degC/s')
ax1.set_xlabel('h [W/m2.K]')
ax1.set_ylabel('Vitesse de refroidissement [degC/s]')
ax1.set_title('Effet de h -- Vitesse de refroidissement')
ax1.legend()

ax2.plot(h_values, dT_maxs, 's-', color='#8e44ad', linewidth=2, markersize=8)
ax2.axhline(200, linestyle='--', color='red', linewidth=1.5, label='Max = 200 degC')
ax2.set_xlabel('h [W/m2.K]')
ax2.set_ylabel('DeltaT_max [degC]')
ax2.set_title('Effet de h -- Gradient thermique maximal')
ax2.legend()

plt.suptitle('Partie D -- Analyse parametrique', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('D_parametric_study.png', dpi=150, bbox_inches='tight')
plt.show()


## 11. Bilan final et recommandation

On cherche la valeur de h qui satisfait **simultanement** les deux contraintes du cahier des charges.

In [ ]:
print('=' * 60)
print('BILAN FINAL -- Recommandation')
print('=' * 60)

fluides = {
    500:  'Huile',
    1000: 'Eau stagnante',
    3000: 'Eau agitee moderement',
    5000: 'Eau agitee violemment'
}

for h, cr, dt_max in zip(h_values, cool_rates, dT_maxs):
    ok_m = cr >= 30
    ok_s = dt_max <= 200
    if ok_m and ok_s:
        statut = '[OPTIMAL] Satisfait les DEUX contraintes'
    elif not ok_m:
        statut = '[ECHEC] Refroidissement trop LENT -- piece trop molle'
    else:
        statut = '[ECHEC] Choc thermique trop FORT -- risque de fissure'
    print(f'\n  h = {h:5d} W/m2.K  [{fluides[h]}]')
    print(f'    Vitesse = {cr:.1f} degC/s  |  DeltaT_max = {dt_max:.1f} degC')
    print(f'    {statut}')

print('\n' + '=' * 60)
print('CONCLUSION')
print('=' * 60)
print()
print('Une agitation moderee du fluide (h ~ 3000 W/m2.K) est la seule')
print('condition qui permet d obtenir la durete voulue au centre de la')
print('piece sans risquer de la fissurer par choc thermique.')
print()
print('  -> Vitesse de refroidissement : verifiee (>= 30 degC/s)')
print('  -> Gradient thermique maximal : verifie (<= 200 degC)')
